# TP 2 - Introduction to Python, RDKit & Molecular Similarity

**Practical Cheminformatics**  
University: Université Ibn Zohr - FSA Aït Melloul  
Semester: 6th Semester (Applied Chemistry)  
Duration: ~4h

Fill your information:
- Full name:
- Apogee code:
- Group:

## Objectives
By the end of this session, you will be able to:
- Open and run code in a Jupyter/Colab notebook
- Write basic Python code (variables, functions, lists)
- Use RDKit to create molecules from SMILES
- Generate InChI and InChIKey programmatically
- Compute molecular descriptors
- Calculate molecular fingerprints
- Compute Tanimoto similarity between molecules

## Part 1 - Getting Started with Python
### Step 5-6: First code

In [ ]:
print("Hello, Cheminformatics!")

**Q1. What output did you see below the cell?**

Your answer: ...

### Step 7: Python basics - variables

In [ ]:
# This is a comment (Python ignores lines starting with #)
molecule_name = "Aspirin"
molecular_weight = 180.16
num_atoms = 13
is_drug = True

print("Molecule:", molecule_name)
print("MW:", molecular_weight, "g/mol")
print("Heavy atoms:", num_atoms)
print("Is a drug?", is_drug)

**Q2. Modify the cell for Caffeine (MW = 194.19, 14 heavy atoms). What output do you get?**

Your answer: ...

### Step 8: Python basics - lists

In [ ]:
# A list of drug names
drugs = ["Aspirin", "Ibuprofen", "Caffeine", "Paracetamol"]

print("Number of drugs:", len(drugs))
print("First drug:", drugs[0])
print("Last drug:", drugs[-1])

for drug in drugs:
    print("  -", drug)

**Q3. Add Diclofenac and Naproxen to the list. How many drugs are there now?**

Your answer: ...

## Part 2 - Installing and Using RDKit
### Step 9: Install RDKit

In [ ]:
!pip install rdkit-pypi
# if it doesn't work, use:
# !pip install rdkit

### Step 10: Create your first molecule

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

first_smiles = "CC(=O)Oc1ccccc1C(=O)O"
mol = Chem.MolFromSmiles(first_smiles)

Draw.MolToImage(mol, size=(300, 300))

**Q4. Do you recognise the molecule displayed? What is it?**

Your answer: ...

### Step 11: Create multiple molecules

In [ ]:
molecules = {
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Caffeine": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "Diclofenac": "OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl",
    "Naproxen": "COc1ccc2cc(ccc2c1)C(C)C(=O)O"
}

mols = []
names = []
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        mols.append(mol)
        names.append(name)
        print(f"✓ {name}: {smi}")
    else:
        print(f"✗ {name}: INVALID SMILES!")

**Q5. Were all molecules created successfully? What does `MolFromSmiles` return if SMILES is invalid?**

Your answer: ...

### Step 12: Visualise molecules

In [ ]:
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), legends=names)

**Q6. Identify each molecule from its 2D structure. Which ones have aromatic rings?**

Your answer: ...

## Part 3 - SMILES, InChI, and InChIKey
### Step 13: Canonical SMILES

In [ ]:
smiles_variants = [
    "CC(=O)Oc1ccccc1C(=O)O",
    "O=C(O)c1ccccc1OC(C)=O",
    "c1ccc(c(c1)OC(=O)C)C(=O)O",
    "OC(=O)c1ccccc1OC(=O)C"
]

print("Original SMILES → Canonical SMILES")
print("-" * 60)
for smi in smiles_variants:
    mol = Chem.MolFromSmiles(smi)
    canonical = Chem.MolToSmiles(mol)
    print(f"  {smi}")
    print(f"  → {canonical}")
    print()

**Q7. Do all four SMILES produce the same canonical SMILES? What does this tell you?**

Your answer: ...

### Step 14: Generate InChI and InChIKey

In [ ]:
from rdkit.Chem.inchi import MolFromInchi, MolToInchi, InchiToInchiKey

print(f"{'Molecule':<15} {'Canonical SMILES':<45} {'InChIKey'}")
print("=" * 100)

for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    canonical = Chem.MolToSmiles(mol)
    inchi = MolToInchi(mol)
    inchikey = InchiToInchiKey(inchi)
    print(f"{name:<15} {canonical:<45} {inchikey}")

**Q8. Copy the InChIKey for Ibuprofen and search it on Google. What do you find?**

Your answer: ...

### Step 15: Verify cross-database consistency

In [ ]:
smiles_pubchem = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
smiles_chembl = "CC(c1ccc(CC(C)C)cc1)C(O)=O"

mol1 = Chem.MolFromSmiles(smiles_pubchem)
mol2 = Chem.MolFromSmiles(smiles_chembl)

canon1 = Chem.MolToSmiles(mol1)
canon2 = Chem.MolToSmiles(mol2)

inchi1 = MolToInchi(mol1)
inchi2 = MolToInchi(mol2)
key1 = InchiToInchiKey(inchi1)
key2 = InchiToInchiKey(inchi2)

print("PubChem SMILES (canonical):", canon1)
print("ChEMBL SMILES (canonical): ", canon2)
print("Same canonical SMILES?", canon1 == canon2)
print()
print("PubChem InChIKey:", key1)
print("ChEMBL InChIKey: ", key2)
print("Same InChIKey?", key1 == key2)

**Q9. Are the canonical SMILES and InChIKeys identical? What does this confirm?**

Your answer: ...

**Q10. Repeat with another molecule from TP1. Do you get the same InChIKey?**

Your answer: ...

## Part 4 - Molecular Descriptors
### Step 16: Compute descriptors

In [ ]:
from rdkit.Chem import Descriptors

print(f"{'Molecule':<15} {'MW':>8} {'LogP':>8} {'HBD':>5} {'HBA':>5} {'TPSA':>8} {'RotBonds':>9}")
print("=" * 70)

for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rotbonds = Descriptors.NumRotatableBonds(mol)

    print(f"{name:<15} {mw:>8.2f} {logp:>8.2f} {hbd:>5d} {hba:>5d} {tpsa:>8.2f} {rotbonds:>9d}")

**Q11. Which molecule has the highest LogP and which has the lowest?**

Your answer: ...

**Q12. Check Lipinski Rule of Five for each molecule.**

| Molecule | MW ≤ 500? | HBD ≤ 5? | HBA ≤ 10? | LogP ≤ 5? | Passes Ro5? |
|---|---|---|---|---|---|
| Aspirin |  |  |  |  |  |
| Ibuprofen |  |  |  |  |  |
| Caffeine |  |  |  |  |  |
| Paracetamol |  |  |  |  |  |
| Diclofenac |  |  |  |  |  |
| Naproxen |  |  |  |  |  |

### Step 17: Sort molecules by LogP

In [2]:
data = []
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    logp = Descriptors.MolLogP(mol)
    data.append((name, logp))

data.sort(key=lambda x: x[1])

print("Molecules sorted by LogP (hydrophilic → lipophilic):")
for name, logp in data:
    bar = "█" * int((logp + 2) * 5)
    print(f"  {name:<15} LogP = {logp:>6.2f}  {bar}")

NameError: name 'molecules' is not defined

**Q13. Does the ordering match your chemical intuition? Which structural features increase lipophilicity?**

Your answer: ...

### Additional analysis: PCA of molecular descriptors

Use the descriptors calculated above to perform **Principal Component Analysis (PCA)** and visualize the molecules in **2D** using **PC1** and **PC2**.



The arrows represent the contribution of each descriptor to the principal components (loadings).


In [ ]:
import pandas as pd

from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler



descriptor_rows = []

for name, smi in molecules.items():

    mol = Chem.MolFromSmiles(smi)

    descriptor_rows.append(

        {

            "Molecule": name,

            "MW": Descriptors.MolWt(mol),

            "LogP": Descriptors.MolLogP(mol),

            "HBD": Descriptors.NumHDonors(mol),

            "HBA": Descriptors.NumHAcceptors(mol),

            "TPSA": Descriptors.TPSA(mol),

            "RotBonds": Descriptors.NumRotatableBonds(mol),

        }

    )



descriptor_df = pd.DataFrame(descriptor_rows).set_index("Molecule")

X_scaled = StandardScaler().fit_transform(descriptor_df)



pca = PCA(n_components=2)

scores = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_ * 100



pca_df = pd.DataFrame(scores, columns=["PC1", "PC2"], index=descriptor_df.index)

loadings = pd.DataFrame(pca.components_.T, columns=["PC1", "PC2"], index=descriptor_df.columns)



print("Descriptor matrix:")

display(descriptor_df.round(2))

print("\nPCA scores:")

display(pca_df.round(3))

print(f"Explained variance -> PC1: {explained_var[0]:.2f}% | PC2: {explained_var[1]:.2f}%")



fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(pca_df["PC1"], pca_df["PC2"], s=90, color="royalblue")



for molecule, row in pca_df.iterrows():

    ax.text(row["PC1"] + 0.05, row["PC2"] + 0.05, molecule, fontsize=10)



score_scale = max(pca_df.abs().max())

arrow_scale = score_scale * 0.9 if score_scale > 0 else 1.0



for descriptor, row in loadings.iterrows():

    ax.arrow(

        0,

        0,

        row["PC1"] * arrow_scale,

        row["PC2"] * arrow_scale,

        color="darkred",

        width=0.01,

        head_width=0.12,

        length_includes_head=True,

        alpha=0.8,

    )

    ax.text(

        row["PC1"] * arrow_scale * 1.12,

        row["PC2"] * arrow_scale * 1.12,

        descriptor,

        color="darkred",

        fontsize=10,

        fontweight="bold",

    )



ax.axhline(0, color="gray", linestyle="--", linewidth=1)

ax.axvline(0, color="gray", linestyle="--", linewidth=1)

ax.set_xlabel(f"PC1 ({explained_var[0]:.1f}% variance)")

ax.set_ylabel(f"PC2 ({explained_var[1]:.1f}% variance)")

ax.set_title("PCA of Molecular Descriptors (2D)")

ax.grid(alpha=0.3)

plt.tight_layout()

plt.show()


## Part 5 - Molecular Fingerprints & Similarity
### Step 18: Compute MACCS keys

In [ ]:
from rdkit.Chem import MACCSkeys

fps = {}
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    fp = MACCSkeys.GenMACCSKeys(mol)
    fps[name] = fp

    bits = fp.ToBitString()
    num_on = bits.count('1')
    print(f"{name:<15} Bits ON: {num_on:>3}/167  First 50 bits: {bits[:50]}...")

**Q14. Which molecule has the most bits set to 1? What does this mean structurally?**

Your answer: ...

### Step 19: Tanimoto similarity matrix

In [ ]:
from rdkit import DataStructs

drug_names = list(fps.keys())

print(f"\n{'Tanimoto Similarity Matrix':^70}")
print(f"{'':>15}", end="")
for name in drug_names:
    print(f"{name:>12}", end="")
print()
print("-" * (15 + 12 * len(drug_names)))

for name1 in drug_names:
    print(f"{name1:>15}", end="")
    for name2 in drug_names:
        sim = DataStructs.TanimotoSimilarity(fps[name1], fps[name2])
        print(f"{sim:>12.3f}", end="")
    print()

**Q15. Which pair is the most similar (excluding self-comparisons)?**

Your answer: ...

**Q16. Which pair is the least similar? Does it make chemical sense?**

Your answer: ...

**Q17. Using thresholds T ≥ 0.85 and T ≥ 0.70, which pairs are considered similar?**

Your answer: ...

### Step 20: Heatmap visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = len(drug_names)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = DataStructs.TanimotoSimilarity(fps[drug_names[i]], fps[drug_names[j]])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(drug_names, rotation=45, ha='right')
ax.set_yticklabels(drug_names)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i][j]:.2f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, label='Tanimoto Similarity')
plt.title('Molecular Similarity Heatmap (MACCS Keys)')
plt.tight_layout()
plt.show()

**Q18. From the heatmap, identify clusters of similar molecules. Do they share common structural features?**

Your answer: ...

### Step 21: Compare MACCS vs Morgan fingerprints

In [ ]:
from rdkit.Chem import AllChem

mol_a = Chem.MolFromSmiles(molecules["Aspirin"])
mol_b = Chem.MolFromSmiles(molecules["Naproxen"])

fp_maccs_a = MACCSkeys.GenMACCSKeys(mol_a)
fp_maccs_b = MACCSkeys.GenMACCSKeys(mol_b)
tani_maccs = DataStructs.TanimotoSimilarity(fp_maccs_a, fp_maccs_b)

fp_morgan_a = AllChem.GetMorganFingerprintAsBitVect(mol_a, radius=2, nBits=2048)
fp_morgan_b = AllChem.GetMorganFingerprintAsBitVect(mol_b, radius=2, nBits=2048)
tani_morgan = DataStructs.TanimotoSimilarity(fp_morgan_a, fp_morgan_b)

print("Aspirin vs Naproxen:")
print(f"  MACCS Keys Tanimoto:           {tani_maccs:.3f}")
print(f"  Morgan (ECFP4) Tanimoto:       {tani_morgan:.3f}")

**Q19. Are Tanimoto values the same for MACCS and Morgan fingerprints? Why might they differ?**

Your answer: ...

### Step 22: Your own similarity analysis
Add 3 new molecules of your choice to `molecules` and re-run fingerprint + similarity cells.

| New Molecule | SMILES | Most Similar To | Tanimoto Score |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

**Q20. List your 3 molecules and the closest existing match.**

Your answer: ...

## Part 6 - Wrap-up

**Q21. List three advantages of using RDKit over manual database searches.**

Your answer: ...

**Q22. Explain in your own words what a molecular fingerprint represents and why it is useful.**

Your answer: ...

**Q23. Which fingerprint type would you prefer for similarity search (MACCS or Morgan), and why?**

Your answer: ...